In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector


load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
  embeddings=embedding_model,
  connection=connection_url,
  collection_name="library_rules",
  use_jsonb=True # 메타데이터를 JSONB로 저장  
)

In [6]:
library_rule_chunks = [
    "[2026학년도 중앙도서관 대출 및 열람 규정]\n제1조(대출자격) 본교 재학생 및 휴학생은 학생증을 제시한 후 도서를 대출할 수 있습니다.",
    "제2조(대출 권수 및 기간) 학부생은 1인당 최대 5권까지 대출할 수 있으며 대출 기간은 14일입니다.\n단, 학술 연구용 특수 도서는 7일간 대출 가능하며 연장은 불가능합니다.",
    "제3조(연체 및 제재) 반납 기한을 초과할 경우 연체 일수만큼 대출이 정지됩니다."
]

chunk_ids = ["rule_chunk_001", "rule_chunk_002", "rule_chunk_003"]

saved_ids = vector_store.add_texts(
    texts=library_rule_chunks,
    ids=chunk_ids
)

print(f"저장 갯수: {len(saved_ids)}")
print(f"청크 ID: {saved_ids}")

저장 갯수: 3
청크 ID: ['rule_chunk_001', 'rule_chunk_002', 'rule_chunk_003']


In [9]:
question = "특수 도서는 며칠 동안 대출할 수 있나요?"

search_results = vector_store.similarity_search(
    query=question,
    k=1 # Top-K 설정
)

print(search_results[0].page_content)

제2조(대출 권수 및 기간) 학부생은 1인당 최대 5권까지 대출할 수 있으며 대출 기간은 14일입니다.
단, 학술 연구용 특수 도서는 7일간 대출 가능하며 연장은 불가능합니다.


In [11]:
search_results2 = vector_store.similarity_search_with_score(
    query=question,
    k=3
)

for chunk, score in search_results2:
    print(f"score={score:.4f} / {chunk.page_content}")

score=0.4440 / 제2조(대출 권수 및 기간) 학부생은 1인당 최대 5권까지 대출할 수 있으며 대출 기간은 14일입니다.
단, 학술 연구용 특수 도서는 7일간 대출 가능하며 연장은 불가능합니다.
score=0.5775 / [2026학년도 중앙도서관 대출 및 열람 규정]
제1조(대출자격) 본교 재학생 및 휴학생은 학생증을 제시한 후 도서를 대출할 수 있습니다.
score=0.7499 / 제3조(연체 및 제재) 반납 기한을 초과할 경우 연체 일수만큼 대출이 정지됩니다.
